### Word2Vec
- 문자를 수치형으로 변환시켜주는 딥러닝 기반의 임베딩 기술
- 매개변수
    - sentences
        - 기본값 : None
        - 토큰화가 된 문장 데이터 (2차원 데이터)
        - None 기본값 : 학습을 시킬 수 있다.
    - vector_size
        - 기본값 : 100
        - 임베딩 벡터 차원의 개수 (feature의 수)
    - window
        - 기본값 : 5
        - 예측 시 고려할 주변 단어와의 거리 (문맥의 크기)
    - sg
        - 기본갑 : 0
        - 0인 경우 : CBOW 방식 (주변 단어들을 이용하여 중심 단어를 예측)
        - 1인 경우 : Skip_gram 방식 (중심 단어를 이용하여 주변 단어를 예측)
    - min_count
        - 기본값 : 5
        - 최소 등장 빈도 수
        - 적게 등장한 단어들을 제외
    - hs
        - 기본값 : 0
        - 계산의 방식 지정
        - 0 : Negative Sampling (계산량 적음)
        - 1 : Hierarchical Softmax (계산량 많음)
    - epochs
        - 기본값 : 100
        - 반복 학습 횟수를 지정
    - max_vocab_size
        - 기본값 : None
        - 메모리 제한 시 사용할 최대 단어의 수
- 속성
    - wv
        - 학습된 단어 벡터 (class 형태로 출력)
        - 예 : model.wv['단어']
    - wv.index_to_key
        - 단어의 리스트 (학습이 된 단어의 개수) -> 최소 등장 횟수에 영항을 줌
        - 등장 빈도 수에 따라 자동 결정
    - wv.key_to_index
        - 단어 -> 인덱스로 매칭
        - 특정 단어가 인덱스 몇에 위치하는가
    - copus_total_word : 전체 학습이 된 단어의 개수
    - epochs : 학습의 epoch 수
    - vector_size : 벡터 차원의 수
- 메서드
    - wv.most_similar(word, topn = 10)
        - 특정 단어와 유사한 단어를 출력
        - topn은 유사한 단어의 개수를 출력
    - wv.similarity(word1, word2) : 두 단어 간의 코사인 유사도
    - wv.get_vector(word) : 특정 단어를 벡터로 변환
    - train() : 추가 데이터로 학습
    - save() : 학습된 모델을 저장
    - Word2Vec.load() : 저장되어있는 모델을 로드

In [ ]:
#라이브러리 설치
# !pip install gensim

: 

In [ ]:
from gensim.models import Word2Vec

In [ ]:
from sklearn.svm import LinearSVC
from konlpy.tag import Komoran

In [ ]:
docs=[
    '오늘 날씨가 좋다. 여행 가고 싶다.',
    '기온이 너무 올라서 아무것도 하기 싫다',
    '수업이 너무 지루하고 졸리다',
    '음식이 너무 맛이 없고 서비스도 별로다.',
    '영화가 너무 재미있어서 시간이 가는 줄 몰랐다'
]
target=[1, 0, 0, 0, 1]

In [ ]:
#Word2Vec는 토크환 된 데이터가 필요
komoran=Komoran()

allow_pos=['NNP','NNG','VV','VA','Sl','MAG']

tokens=[]

for doc in docs:
    words=[]
    for word, pos in komoran.pos(doc):
        if pos in allow_pos:
            words.append(word)
    tokens.append(words)

tokens

In [ ]:
#Word2Vec를 이용히여 학습 (Skip-gram 방식)
w2v=Word2Vec(
    sentences=tokens,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1,
    epochs=100,
    seed=42,
    workers=2
)

In [ ]:
#Word2Vec에서 wv속성은 ㅇ객치(class)로 뱐환 -> 자주 사용되는 객체임으로 변수에 저장
wv=w2v.wv

In [ ]:
#wv에 특정 단어를 입력하면 벡터 출력
wv['여행']

In [ ]:
#유사한 단어 찾기
wv.most_similar('음식',topn=3)

In [ ]:
#두 단어의 코사인 유사도를 확인
wv.similarity('음식','여행')

In [ ]:
import numpy as np

In [ ]:
from sklearn import set_config
#모델과 디스플레이방식을 html 다이어그렘에서 텍스트로 변경
set_config(display='text')

In [ ]:
tokens[0]

In [ ]:
#tokens의 단어들 중 w2v의 index_to_key에 존재하는 데이터의 단위 벡터를 확인 

#vectors : docs의 문장들을 벡터화한 리스트
vectors = []

for token in tokens:
    vec = []
    for word in token:
        #print(word)
        # vec = []    ## 문제점? : token의 각 원소를 word에 대입하여 반복 실행하면서 매변 초기화 -> 마지막 단어의 벡터값만 vec에 대입
        if word in wv.index_to_key:
            # print(word)
            #tokens 데이터에서 단어가 w2v의 학습 단어에 포함되어있을때
            #해당 단어의 벡터 값을 vec에 추가 
            vec.append(wv[word])
            # print(wv[word].shape)
    print(np.array(vec).shape)

    vectors.append( np.mean(vec, axis=0) )
    #     break
    # break    
print(vectors)

In [ ]:
np.array(vectors).shape

In [ ]:
svc=LinearSVC(random_state=42)

#decode error 발생시
#C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_repr_html\estimator.py 오픈
#560번 라인 근처
#with open(str(Path(__file__).parent/'estimator.py'),'r') as f: 이 부분을
#encoding = 'utf-8' 추가
#with open(str(Path(__file__).parent/'estimator.py'),'r', encoding = 'utf-8') as f: 수정 후 저장

svc.fit(vectors, target)

In [ ]:
svc.predict(vectors)